# v4.3.2 — NextEmbedHead 학습 (Colab self-contained)

**무엇**: Sentence-T5 (frozen) 위에 작은 MLP head 를 학습해서, 주어진 5-turn causal context 의 embedding 으로부터 *다음 발화의 ST5 embedding* 을 cosine-regression 으로 예측. 학습된 head 는 Hi-OnTop v4.3.2 segmenter 의 δ_model 로 사용.

**왜 self-contained**: setup_colab.ipynb / Hi-OnTop repo clone 없이 이 notebook 한 개만으로 학습 → ckpt 산출. ckpt 만 로컬 `outputs/runs/_misc/next_embed_head_<tag>.pt` 로 옮기면 됨.

**사용 흐름**:
1. `ijcnlp_dailydialog.zip` 을 Colab session 에 업로드 (좌측 file browser 또는 `[1b]` 셀).
2. 모든 셀 순서대로 실행.
3. `[8]` 의 ckpt 를 `[9]` 셀에서 다운로드.

**참고**: `colab_csm_train.ipynb` 의 `[2]` 셀 (zip 자동 탐색 + 재귀 추출) 과 동일 패턴 사용.

In [ ]:
# [1] 환경 / 의존성
import os, sys, subprocess, time, math, glob, zipfile, shutil, json, pickle

try:
    import google.colab as _gc  # noqa
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

PROJECT_ROOT = '/content' if IS_COLAB else os.path.abspath(os.getcwd())
OUT_DIR = os.path.join(PROJECT_ROOT, 'v432_artifacts')
os.makedirs(OUT_DIR, exist_ok=True)
print('IS_COLAB =', IS_COLAB)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUT_DIR      =', OUT_DIR)

if IS_COLAB:
    # numpy 핀 안 함 — Colab default (numpy 2.x) 와 torch/sentence-transformers ABI 충돌 회피.
    # 우리 코드는 numpy 2.x 호환.
    INSTALLED_FLAG = '/tmp/.v432_installed'
    if not os.path.exists(INSTALLED_FLAG):
        print('[install] first run — installing deps + restarting runtime...')
        !pip -q install 'torch>=2.0' 'transformers>=4.40,<4.50' 'sentence-transformers>=3.0,<4.0' 2>&1 | tail -3
        open(INSTALLED_FLAG, 'w').write('1')
        print('\n==== install 완료. 런타임 자동 재시작 (정상). 셀 [1] 부터 다시 실행하세요. ====')
        os.kill(os.getpid(), 9)   # 강제 재시작
    else:
        print('[install] already installed (flag), skip')

In [ ]:
# [1b] (선택) ijcnlp_dailydialog.zip Colab 업로드
# 이미 PROJECT_ROOT 에 zip 있으면 이 셀 skip.
if IS_COLAB:
    from google.colab import files
    up = files.upload()  # → /content/<filename>
    for name in up.keys():
        print('uploaded:', name)

In [ ]:
# [2] DailyDialog 원본 준비 — PROJECT_ROOT 에서 zip 자동 탐색 + 재귀 추출
#   원본 zip 안: ijcnlp_dailydialog/{dialogues_text,topic,act,emotion}.txt
#   추가로 안에 train.zip / validation.zip / test.zip 이 있으면 그것도 추출.
DD_DIR = os.path.join(PROJECT_ROOT, 'dailydialog')
EXTRACT_DIR = os.path.join(PROJECT_ROOT, 'dd_extract')
os.makedirs(DD_DIR, exist_ok=True)
EOU = '__eou__'
SPLIT_FILES = {
    'train': 'dialogues_train.txt',
    'validation': 'dialogues_validation.txt',
    'test': 'dialogues_test.txt',
}

def have_any_dd():
    for f in ['dialogues_text.txt', *SPLIT_FILES.values()]:
        p = os.path.join(DD_DIR, f)
        if os.path.isfile(p) and os.path.getsize(p) > 0:
            return True
    return False

if not have_any_dd():
    raw = (glob.glob(os.path.join(PROJECT_ROOT, '**', 'ijcnlp_dailydialog*.zip'), recursive=True) +
           glob.glob(os.path.join(PROJECT_ROOT, '**', '*dailydialog*.zip'), recursive=True) +
           glob.glob(os.path.join(PROJECT_ROOT, '*.zip')))
    cand = []
    for p in dict.fromkeys(raw):
        if zipfile.is_zipfile(p):
            cand.append(p)
        else:
            print('skip invalid zip:', p)
    if not cand:
        raise SystemExit(f'!! ijcnlp_dailydialog*.zip 없음. {PROJECT_ROOT}/ 에 업로드 후 재실행.')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print('extract:', cand[0], '->', EXTRACT_DIR)
    with zipfile.ZipFile(cand[0]) as z:
        z.extractall(EXTRACT_DIR)
    for zp in glob.glob(os.path.join(EXTRACT_DIR, '**', '*.zip'), recursive=True):
        if zipfile.is_zipfile(zp):
            print('  (inner) extract', zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(os.path.dirname(zp))
    # 필요한 파일 DD_DIR 로 복사
    for name in ['dialogues_text.txt', *SPLIT_FILES.values()]:
        hits = sorted(
            glob.glob(os.path.join(EXTRACT_DIR, '**', name), recursive=True),
            key=lambda p: (('train' in p) + ('valid' in p) + ('test' in p), len(p)),
        )
        if hits:
            dst = os.path.join(DD_DIR, name)
            if not (os.path.isfile(dst) and os.path.getsize(dst) > 0):
                shutil.copy(hits[0], dst)
                print('placed', name, '<-', hits[0])

print('DD_DIR contents:', sorted(os.listdir(DD_DIR)) if os.path.isdir(DD_DIR) else '(missing)')

In [ ]:
# [3] DailyDialog parse → {split: list[list[str]]}
import numpy as np

def parse_dd_file(path):
    out = []
    with open(path, 'r', encoding='utf-8') as fh:
        for line in fh:
            parts = [u.strip() for u in line.split(EOU)]
            parts = [u for u in parts if u]
            if len(parts) >= 2:
                out.append(parts)
    return out

splits = {}
have_official = all(os.path.isfile(os.path.join(DD_DIR, fn)) for fn in SPLIT_FILES.values())
if have_official:
    for k, fn in SPLIT_FILES.items():
        splits[k] = parse_dd_file(os.path.join(DD_DIR, fn))
    print('[data] official split files used')
else:
    all_path = os.path.join(DD_DIR, 'dialogues_text.txt')
    all_dialogs = parse_dd_file(all_path)
    rng = np.random.default_rng(0)
    idx = rng.permutation(len(all_dialogs))
    n = len(all_dialogs); n_va = max(1, n // 20); n_te = max(1, n // 20); n_tr = n - n_va - n_te
    splits['train'] = [all_dialogs[i] for i in idx[:n_tr]]
    splits['validation'] = [all_dialogs[i] for i in idx[n_tr:n_tr+n_va]]
    splits['test'] = [all_dialogs[i] for i in idx[n_tr+n_va:]]
    print(f'[data] split 파일 없음 → 90/5/5 random split (seed=0)')
for k, v in splits.items():
    print(f'  {k}: n_dial={len(v)}, n_utt={sum(len(d) for d in v)}')

In [ ]:
# [4] NextEmbedHead 정의 (MLP / Transformer 둘 다, factory)
import torch
import torch.nn as nn
import torch.nn.functional as F

class NextEmbedHeadMLP(nn.Module):
    def __init__(self, emb_dim=768, context_window=5, hidden_dim=1024):
        super().__init__()
        self.emb_dim = emb_dim; self.context_window = context_window; self.hidden_dim = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(emb_dim * context_window, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, emb_dim),
        )
    def forward(self, ctx):
        if ctx.dim() == 3: ctx = ctx.flatten(1, 2)
        out = self.net(ctx)
        return F.normalize(out, p=2, dim=-1, eps=1e-12)
    @staticmethod
    def cosine_loss(pred, target):
        cos = (pred * target).sum(dim=-1)
        return (1.0 - cos).mean()

class NextEmbedHeadTransformer(nn.Module):
    """1-layer Transformer encoder + learned pos emb + pad mask + mean pool.

    MLP 의 zero-pad shortcut + concat 의 position 정보 부재 해결 목적.
    """
    def __init__(self, emb_dim=768, context_window=5, n_heads=8,
                 dim_feedforward=1024, n_layers=1, dropout=0.1):
        super().__init__()
        self.emb_dim = emb_dim; self.context_window = context_window
        self.hidden_dim = dim_feedforward
        self.n_heads = n_heads; self.n_layers = n_layers; self.dropout = dropout
        self.pos_embed = nn.Parameter(torch.randn(context_window, emb_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.proj = nn.Linear(emb_dim, emb_dim)
    def forward(self, ctx):
        if ctx.dim() == 2:
            ctx = ctx.view(-1, self.context_window, self.emb_dim)
        pad_mask = ctx.abs().sum(dim=-1) < 1e-6   # (B, m), True=padded
        x = ctx + self.pos_embed
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).float().unsqueeze(-1)
        pooled = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        out = self.proj(pooled)
        return F.normalize(out, p=2, dim=-1, eps=1e-12)

def make_head(head_type, **kw):
    if head_type == 'mlp':
        return NextEmbedHeadMLP(emb_dim=kw.get('emb_dim', 768),
                                context_window=kw.get('context_window', 5),
                                hidden_dim=kw.get('hidden_dim', 1024))
    if head_type == 'transformer':
        return NextEmbedHeadTransformer(emb_dim=kw.get('emb_dim', 768),
                                        context_window=kw.get('context_window', 5),
                                        n_heads=kw.get('n_heads', 8),
                                        dim_feedforward=kw.get('hidden_dim', 1024),
                                        n_layers=kw.get('n_layers', 1),
                                        dropout=kw.get('dropout', 0.1))
    raise ValueError(f'unknown head_type={head_type!r}')

In [ ]:
# [5] Hyperparameters — 셀 단위로 조정
EPOCHS = 50               # 최대 epoch (early stop 이 더 일찍 끊을 수 있음)
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
M = 5                     # causal window
HIDDEN_DIM = 1024
ENC_NAME = 'sentence-transformers/sentence-t5-base'

# ★ head architecture
HEAD_TYPE = 'transformer'  # 'mlp' or 'transformer'
N_HEADS = 8               # transformer only
N_LAYERS = 1              # transformer only
DROPOUT = 0.1             # transformer only

NCE_WEIGHT = 0.0          # collapse 진단 후 조건부
NCE_TEMP = 0.07
PATIENCE = 7              # early stop
SEED = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# tag — head_type 포함해서 ckpt 충돌 회피
if HEAD_TYPE == 'transformer':
    TAG = f'st5_m{M}_tx_h{N_HEADS}_l{N_LAYERS}_ff{HIDDEN_DIM}'
else:
    TAG = f'st5_m{M}_mlp{HIDDEN_DIM}'
print(f'device={DEVICE}, tag={TAG}, head={HEAD_TYPE}, patience={PATIENCE}')

In [ ]:
# [6] Sentence-T5 frozen encode (모든 utterance 1회). 캐시 재사용.
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(SEED); np.random.seed(SEED)
from sentence_transformers import SentenceTransformer

EMB_CACHE = os.path.join(OUT_DIR, f'ijcnlp_dailydialog_emb_{ENC_NAME.split("/")[-1]}.pkl')
if os.path.exists(EMB_CACHE):
    print('[cache] reuse', EMB_CACHE)
    with open(EMB_CACHE, 'rb') as fh:
        embs_per_split = pickle.load(fh)
else:
    print('[encode] loading', ENC_NAME, '(device=' + DEVICE + ')')
    st_model = SentenceTransformer(ENC_NAME, device=DEVICE)
    embs_per_split = {}
    for split, dialogs in splits.items():
        flat, sizes = [], []
        for d in dialogs:
            flat.extend(d); sizes.append(len(d))
        print(f'  [encode] {split}: n_utt={len(flat)}')
        embs = st_model.encode(flat, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
        embs = np.asarray(embs, dtype=np.float32)
        out, i = [], 0
        for n in sizes:
            out.append(embs[i:i+n]); i += n
        embs_per_split[split] = out
    with open(EMB_CACHE, 'wb') as fh:
        pickle.dump(embs_per_split, fh)
    print('  →', EMB_CACHE)
for k, v in embs_per_split.items():
    print(f'  {k}: n_dial={len(v)}, first_emb_shape={v[0].shape if v else None}')

In [ ]:
# [7] (ctx, tgt) pair + diagnostics (4-tier classifier, codex+사용자 검토 기반)
from torch.utils.data import DataLoader, TensorDataset

def build_pairs(embs_list, m):
    ctx_list, tgt_list = [], []
    for emb in embs_list:
        n, d = emb.shape
        if n < 2: continue
        for t in range(1, n):
            start = max(0, t - m); win = emb[start:t]; k = win.shape[0]
            if k < m:
                pad = np.zeros((m - k, d), dtype=emb.dtype)
                win = np.concatenate([pad, win], axis=0)
            ctx_list.append(win); tgt_list.append(emb[t])
    return np.stack(ctx_list, axis=0), np.stack(tgt_list, axis=0)

def make_loader(ctx, tgt, bs, shuffle):
    ctx_t = torch.from_numpy(ctx).float()
    tgt_t = F.normalize(torch.from_numpy(tgt).float(), p=2, dim=-1, eps=1e-12)
    return DataLoader(TensorDataset(ctx_t, tgt_t), batch_size=bs, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

def mixed_loss(pred, target, nce_weight=0.0, temperature=0.07):
    reg = NextEmbedHeadMLP.cosine_loss(pred, target)
    if nce_weight <= 0.0: return reg
    logits = pred @ target.T / temperature
    labels = torch.arange(pred.shape[0], device=pred.device)
    return reg + nce_weight * F.cross_entropy(logits, labels)

# === 4-tier diagnostic thresholds (codex + 사용자 검토 2026-05-21) ===
THR = {
    'gain_mean_success': 0.035, 'gain_mean_acceptable': 0.020, 'gain_mean_retry': 0.0,
    'gain_last_fail': 0.0, 'gain_last_acceptable': 0.005,
    'gain_full_success': 0.025, 'gain_full_acceptable': 0.010, 'gain_full_retry': 0.0,
    'delta_std_success': 0.045, 'delta_std_acceptable': 0.030, 'delta_std_retry': 0.020,
    'pred_norm_lo': 0.995, 'pred_norm_hi': 1.005,
}

def classify_diag(diag):
    pn = diag.get('pred_norm_mean', 1.0)
    if not (THR['pred_norm_lo'] <= pn <= THR['pred_norm_hi']):
        return 'CRITICAL', [f'pred_norm={pn:.4f} 이탈 (L2-norm 코드 의심)']
    warns, fail = [], False
    g_mean, g_last = diag.get('gain_vs_mean', 0.0), diag.get('gain_vs_last', 0.0)
    g_full, d_std = diag.get('gain_vs_mean_full', 0.0), diag.get('delta_std', 0.0)
    if g_mean <= THR['gain_mean_retry']: warns.append(f'MEAN-COLLAPSE (gain_mean={g_mean:+.4f}≤0)'); fail=True
    if g_last <= THR['gain_last_fail']: warns.append(f'LAST-IDENT (gain_last={g_last:+.4f}≤0)'); fail=True
    if g_full <= THR['gain_full_retry']: warns.append(f'FULL-CTX-COLLAPSE (full_gain={g_full:+.4f}≤0)'); fail=True
    if d_std < THR['delta_std_retry']: warns.append(f'low δ_std ({d_std:.4f}<{THR["delta_std_retry"]})'); fail=True
    if fail: return 'FAIL', warns
    is_success = (g_mean >= THR['gain_mean_success'] and g_full >= THR['gain_full_success']
                  and g_last >= THR['gain_last_acceptable'] and d_std >= THR['delta_std_success'])
    if is_success: return 'SUCCESS', []
    is_retry = (g_mean < THR['gain_mean_acceptable'] or g_full < THR['gain_full_acceptable']
                or d_std < THR['delta_std_acceptable'])
    status = 'RETRY' if is_retry else 'ACCEPTABLE'
    notes = []
    if g_mean < THR['gain_mean_success']: notes.append(f'gain_mean={g_mean:+.4f}<{THR["gain_mean_success"]}')
    if g_full < THR['gain_full_success']: notes.append(f'full_gain={g_full:+.4f}<{THR["gain_full_success"]}')
    if d_std < THR['delta_std_success']: notes.append(f'δ_std={d_std:.4f}<{THR["delta_std_success"]}')
    return status, notes

@torch.no_grad()
def eval_diagnostics(model, loader, device):
    model.eval(); preds, tgts, ctxs = [], [], []
    for ctx, tgt in loader:
        ctx_d, tgt_d = ctx.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        preds.append(model(ctx_d).cpu()); tgts.append(tgt_d.cpu())
        ctxs.append(ctx.cpu() if ctx.device.type != 'cpu' else ctx)
    pred = torch.cat(preds, 0); tgt = torch.cat(tgts, 0); ctx_all = torch.cat(ctxs, 0)
    cos = (pred * tgt).sum(dim=-1); deltas = 1.0 - cos
    mean_tgt = F.normalize(tgt.mean(0, keepdim=True), p=2, dim=-1, eps=1e-12)
    mean_baseline_deltas = 1.0 - (mean_tgt * tgt).sum(dim=-1)
    last_utt = F.normalize(ctx_all[:, -1, :], p=2, dim=-1, eps=1e-12)
    last_baseline_deltas = 1.0 - (last_utt * tgt).sum(dim=-1)
    is_padded = ctx_all[:, 0, :].abs().sum(dim=-1) < 1e-6
    full_mask = ~is_padded
    def _m(t): return float(t.mean()) if t.numel() > 0 else float('nan')
    def _s(t): return float(t.std(unbiased=False)) if t.numel() > 1 else float('nan')
    return {
        'loss': _m(deltas),
        'mean_baseline_loss': _m(mean_baseline_deltas),
        'gain_vs_mean': _m(mean_baseline_deltas) - _m(deltas),
        'last_baseline_loss': _m(last_baseline_deltas),
        'gain_vs_last': _m(last_baseline_deltas) - _m(deltas),
        'delta_std': _s(deltas), 'pred_norm_mean': float(pred.norm(dim=-1).mean()),
        'n_padded': int(is_padded.sum().item()), 'n_full': int(full_mask.sum().item()),
        'loss_full': _m(deltas[full_mask]),
        'mean_baseline_loss_full': _m(mean_baseline_deltas[full_mask]),
        'gain_vs_mean_full': _m(mean_baseline_deltas[full_mask]) - _m(deltas[full_mask]),
        'delta_std_full': _s(deltas[full_mask]),
    }

ctx_tr, tgt_tr = build_pairs(embs_per_split['train'], M)
ctx_va, tgt_va = build_pairs(embs_per_split['validation'], M)
print(f'pairs  train={ctx_tr.shape[0]}  valid={ctx_va.shape[0]}')
train_loader = make_loader(ctx_tr, tgt_tr, BATCH_SIZE, shuffle=True)
valid_loader = make_loader(ctx_va, tgt_va, BATCH_SIZE, shuffle=False)

In [ ]:
# [8] 학습 loop + 4-tier 진단 + early stopping (head_type 분기)
model = make_head(HEAD_TYPE, emb_dim=ctx_tr.shape[-1], context_window=M,
                  hidden_dim=HIDDEN_DIM, n_heads=N_HEADS,
                  n_layers=N_LAYERS, dropout=DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'[model] head={HEAD_TYPE}, params={n_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
def lr_lambda(step):
    if step < warmup_steps: return (step + 1) / max(1, warmup_steps)
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

CKPT_PATH = os.path.join(OUT_DIR, f'next_embed_head_{TAG}.pt')
best_val, best_epoch, patience_counter, history = float('inf'), -1, 0, []
stopped_early, last_ep = False, 0

for ep in range(EPOCHS):
    last_ep = ep
    t0 = time.perf_counter()
    model.train(); losses = []
    for ctx, tgt in train_loader:
        ctx, tgt = ctx.to(DEVICE, non_blocking=True), tgt.to(DEVICE, non_blocking=True)
        pred = model(ctx)
        loss = mixed_loss(pred, tgt, nce_weight=NCE_WEIGHT, temperature=NCE_TEMP)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        losses.append(float(loss.item()))
    tr_loss = float(np.mean(losses))
    diag = eval_diagnostics(model, valid_loader, DEVICE)
    va_loss = diag['loss']; dt = time.perf_counter() - t0
    history.append({'epoch': ep, 'train_loss': tr_loss, **{f'valid_{k}': v for k, v in diag.items()}, 'wall_s': dt})
    marker = ''
    if va_loss < best_val:
        best_val, best_epoch, patience_counter = va_loss, ep, 0
        marker = '  ★ saved'
        torch.save({
            'state_dict': model.state_dict(),
            'config': {
                'head_type': HEAD_TYPE,
                'emb_dim': model.emb_dim, 'context_window': model.context_window,
                'hidden_dim': getattr(model, 'hidden_dim', HIDDEN_DIM),
                'n_heads': N_HEADS, 'n_layers': N_LAYERS, 'dropout': DROPOUT,
                'encoder': ENC_NAME,
            },
            'best_epoch': ep, 'best_valid_loss': va_loss, 'best_valid_diag': diag,
            'args': dict(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                         warmup_frac=WARMUP_FRAC, m=M, hidden_dim=HIDDEN_DIM,
                         head_type=HEAD_TYPE, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
                         nce_weight=NCE_WEIGHT, nce_temp=NCE_TEMP, patience=PATIENCE,
                         seed=SEED, tag=TAG, encoder=ENC_NAME),
        }, CKPT_PATH)
    else:
        patience_counter += 1
    status, notes = classify_diag(diag)
    note_str = ('  ' + ' | '.join(notes)) if notes else ''
    print(f"[ep {ep:02d}] tr={tr_loss:.4f}  va={va_loss:.4f}  "
          f"gain_mean={diag['gain_vs_mean']:+.4f}  gain_last={diag['gain_vs_last']:+.4f}  "
          f"full_gain={diag['gain_vs_mean_full']:+.4f}  δ_std={diag['delta_std']:.4f}  "
          f"pred_norm={diag['pred_norm_mean']:.3f}  "
          f"(n_pad={diag['n_padded']}, n_full={diag['n_full']}, {dt:.1f}s){marker} [{status}]{note_str}")
    if PATIENCE > 0 and patience_counter >= PATIENCE:
        stopped_early = True
        print(f"\n[early stop] valid 가 {PATIENCE} epoch 동안 개선 없음 "
              f"(best ep={best_epoch}, valid={best_val:.4f}). 학습 중단.")
        break

if stopped_early:
    print(f'[summary] stopped at ep {last_ep+1}/{EPOCHS} (early stop)')
print(f'[best] epoch={best_epoch}, valid={best_val:.4f}')
ctx_te, tgt_te = build_pairs(embs_per_split['test'], M)
test_loader = make_loader(ctx_te, tgt_te, BATCH_SIZE, shuffle=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
te_diag = eval_diagnostics(model, test_loader, DEVICE)
te_status, te_notes = classify_diag(te_diag)
te_note_str = ('  ' + ' | '.join(te_notes)) if te_notes else ''
print(f"[test] loss={te_diag['loss']:.4f}  gain_mean={te_diag['gain_vs_mean']:+.4f}  "
      f"gain_last={te_diag['gain_vs_last']:+.4f}  full_gain={te_diag['gain_vs_mean_full']:+.4f}  "
      f"δ_std={te_diag['delta_std']:.4f}  (n_pad={te_diag['n_padded']}, n_full={te_diag['n_full']}) "
      f"[{te_status}]{te_note_str}")
print()
print('=' * 70)
print('⚠ 학습 진단은 *필요조건*. 진짜 판정은 v4.3.2 segmentation sweep')
print('  (`precompute_v432_delta.py` + `run_v432_smoke.py`) 결과로.')
print('=' * 70)

HIST_PATH = CKPT_PATH.replace('.pt', '.history.json')
with open(HIST_PATH, 'w') as fh:
    json.dump({'history': history, 'best_epoch': best_epoch, 'best_valid': best_val,
               'stopped_early': stopped_early, 'last_ep': last_ep,
               'test_diag': te_diag, 'test_status': te_status,
               'config': ckpt['config'], 'args': ckpt['args']}, fh, indent=2)
print('ckpt:', CKPT_PATH); print('hist:', HIST_PATH)

In [ ]:
# [9] ckpt 다운로드 (Colab → 로컬). 로컬 Hi-OnTop repo 의
#     outputs/runs/_misc/ 디렉토리에 그대로 두면 됨.
if IS_COLAB:
    from google.colab import files
    files.download(CKPT_PATH)
    files.download(HIST_PATH)
else:
    print('local mode — files already at:')
    print(' ', CKPT_PATH)
    print(' ', HIST_PATH)